# 09a — Plugin smoke test: no model, one GPU, ~5 GB

Checks the `dg_topk_plugin` code paths **inside your installed vLLM** without starting a server or loading the model, so it can run on any GPU with ~5 GB free (GPUs 0/1 can stay busy).

1. vLLM's plugin loader finds and runs our plugin (`vllm.plugins.load_general_plugins()`), and it patches vLLM's DiffusionGemma module.
2. On real logits and the real embedding table, the patched `sample_row_stats` gives **identical** argmax / samples / entropies to vLLM's original (mode `vllm`), and the patched `_store_self_conditioning` gives a hint within ~0.1% of vLLM's full hint, written into vLLM's state tensor the same way (`keep` and pinned positions respected).
3. Mode `topk_exact`: identical argmax / entropies, valid sampled tokens.
4. A constrained step (logits over a small allowed set, as vLLM uses for structured reads) runs through the patched functions.

Run it in the `.venv-vllm` environment with the plugin installed (RUNBOOK 1b).

In [ ]:
import os, sys, subprocess, importlib
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel first.')
GPU = None      # None = the GPU with the most free memory
if GPU is None:
    rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()
    GPU = str(max(((int(i), int(f)) for i, f in (r.split(', ') for r in rows)), key=lambda t: t[1])[0])
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.pop('DG_TOPK_SC_K', None)          # off while we keep vLLM's original functions
print('using GPU', GPU)

import importlib.metadata as md_
import torch, vllm
import vllm.model_executor.models.diffusion_gemma as dg
print('vllm', vllm.__version__)
eps = [e for e in md_.entry_points(group='vllm.general_plugins') if e.name == 'dg_topk']
print('plugin entry point registered with vLLM:', bool(eps))
assert eps, 'plugin not installed: pip install -e ./dg_topk_plugin'
orig_stats, orig_store = dg.sample_row_stats, dg._store_self_conditioning    # vLLM's originals

os.environ['DG_TOPK_SC_K'] = '64'
os.environ['DG_TOPK_SC_TAIL'] = 'renorm'
os.environ['DG_TOPK_SAMPLE'] = 'vllm'
from vllm.plugins import load_general_plugins
import vllm.plugins as vp
already = vp.plugins_loaded
load_general_plugins()                          # what every vLLM process does at start-up
if not getattr(dg, '_dg_topk_patched', False) and already:
    import dg_topk_plugin; dg_topk_plugin.register()      # plugins were already loaded in this process (with k unset)
print('plugin patched vLLM:', getattr(dg, '_dg_topk_patched', False), '| via vLLM loader:', not already)
assert getattr(dg, '_dg_topk_patched', False), 'the plugin did not patch vLLM'
assert dg.sample_row_stats is not orig_stats and dg._store_self_conditioning is not orig_store
print('register_warmup kept on the patched function:', hasattr(dg.sample_row_stats, 'register_warmup'))

In [ ]:
import hint_bench as hb
CL = 256
saved = hb.load_saved_logits('./runs/nb04/step_*.pt')
assert saved, 'needs the logits saved by notebook 04 in runs/nb04/'
E, _ = hb.load_embedding_table('google/diffusiongemma-26B-A4B-it', 'cuda')
V, D = E.shape
n = 4
L, temps = hb.make_batch(saved, n, CL, 'cuda')
normalizer = torch.tensor(D ** 0.5, dtype=E.dtype, device='cuda')
seed = 1234

a1, s1, e1, p1 = orig_stats(L, temps, CL, seed, E.dtype)
a2, s2, e2, p2 = dg.sample_row_stats(L, temps, CL, seed, E.dtype)
print('mode vllm | argmax identical:', bool(torch.equal(a1, a2)), '| sample identical:', bool(torch.equal(s1, s2)),
      '| entropy max |diff|:', float((e1 - e2).abs().max()), '| probs type:', type(p2).__name__)

def run_store(store, probs, keep, pin):
    sc = torch.zeros(n, CL, D, dtype=E.dtype, device='cuda')
    store(sc, torch.arange(n, device='cuda'), keep, probs.view(n, CL, -1), pin, E, normalizer, 0, V, 1, '')
    return sc.float()
keep = torch.tensor([True, True, False, True], device='cuda')           # slot 2 finishes: its hint must be zero
pin = torch.zeros(n, CL, dtype=torch.bool, device='cuda'); pin[0, :8] = True   # pinned positions: hint must be zero
h1 = run_store(orig_store, p1, keep, pin)
h2 = run_store(dg._store_self_conditioning, p2, keep, pin)
live = keep[:, None] & ~pin
rel = (h2 - h1).norm(dim=-1)[live] / h1.norm(dim=-1)[live].clamp_min(1e-6)
print(f'hint vs vLLM full hint: mean {100 * float(rel.mean()):.3f}%  p99 {100 * float(rel.quantile(0.99)):.3f}%')
print('zero where vLLM writes zero (finished slot, pinned positions):', bool((h2[~live] == 0).all()), bool((h1[~live] == 0).all()))

In [ ]:
# mode topk_exact: reload the module (clears the patch), switch the mode, register again
os.environ['DG_TOPK_SAMPLE'] = 'topk_exact'
dg = importlib.reload(dg)
dg._dg_topk_patched = False                  # reload keeps old module attributes; clear the flag
import dg_topk_plugin; dg_topk_plugin.register()
a3, s3, e3, p3 = dg.sample_row_stats(L, temps, CL, seed, E.dtype)
print('mode topk_exact | argmax identical:', bool(torch.equal(a1, a3)), '| entropy max |diff|:', float((e1 - e3).abs().max()),
      '| samples in vocab:', bool(((s3 >= 0) & (s3 < V)).all()),
      '| samples equal to vLLM\'s Gumbel samples (expected: mostly not):', f'{float((s3 == s1).float().mean()):.2f}')
a4, s4, e4, p4 = dg.sample_row_stats(L, temps, CL, seed, None)             # a step with no self-conditioning
print('no-self-conditioning step returns probs=None:', p4 is None)

# constrained step: logits over a small allowed set of 16 ids
Lc = L[:, :16].contiguous()
ac, sc_, ec, pc = dg.sample_row_stats(Lc, temps, CL, seed, E.dtype)
hc = run_store(dg._store_self_conditioning, pc, torch.ones(n, dtype=torch.bool, device='cuda'), torch.zeros(n, CL, dtype=torch.bool, device='cuda'))
print('constrained (16 ids): runs, samples in range:', bool(((sc_ >= 0) & (sc_ < 16)).all()), '| hint finite:', bool(torch.isfinite(hc).all()))
print('\nSmoke test finished. If every line above is True / small, the plugin is ready for notebook 09.')